# 02 · Crop diagnostic: why edited crops fail (Milestone 2, 14 Nov 2026)

CP1 run 4 drew 3,000 edited training crops (DiffSeg30k) with the CP1 rule (a uniform 256 window,
edited share 1–90%, 10 tries); **1,585 failed**. Before the 1% minimum is changed, this notebook
reports why: it rebuilds exactly the same draws from run 4's manifest
(`Paper 1/runs/checkpoints/cp1/manifest_cp1.csv.gz`) and records the edited share of every try.

**What it does**
1. Copies DiffSeg30k to the local disk (the masks; about 26 GB, the same step as Cell 3 of
   notebook 01, only for DiffSeg30k). In a runtime where notebook 01 already ran, it is skipped.
2. Rebuilds run 4's edited draws (seed 42) and checks that 1,585 of them fail again; then reports
   the failed draws by kind (all tries below 1%, all above 90%, mixed), the share histogram of
   their tries, and how the new training rule does on the same draws (with the 1% minimum and with
   a minimum of 256 edited pixels). It also checks every edited training image over every window:
   how many can never give a valid crop.

**Result** (send this file to Claude): `Paper 1/runs/checkpoints/crop_diagnostic.json`.
CPU runtime; no GPU. Secrets: `GITHUB_TOKEN` only. Nothing is decided from it until Dhanush answers.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
import json, logging, os, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import get_paths, session_info

WORKERS = os.cpu_count() or 2
RUN4_FAILED = 1585        # edited crop draws that failed in CP1 run 4 (docs/CHECKPOINTS.md)
T0 = time.time()
info = session_info()
paths = get_paths()
if DRY:   # a fresh fake Drive of its own
    import shutil, tempfile
    from pathlib import Path
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_crop_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
MANIFEST = paths.runs / 'checkpoints' / 'cp1' / 'manifest_cp1.csv.gz'
print('commit', info['git_commit'], '| workers', WORKERS, '| manifest', MANIFEST)

In [ ]:
# Cell 3: DiffSeg30k masks on the local disk (dry run: a fake world and a fake CP1 run instead)
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    W = dict(sd14_per_class=30, val_per_class=2, diffseg=(36, 12),
             sid_caps={'train': 24, 'validation': 6}, coco_images=60)
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', **W)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True,
                overrides=cp1_overrides(W['coco_images']))
    CROPS = 150
    dry_cfg = CP1Config(workers=2, crops_per_class=CROPS, blockiness_per_group=40,
                        diffseg_card_splits={'train': W['diffseg'][0], 'validation': W['diffseg'][1]})
    dry = run_cp1(paths, dry_cfg, out_dir=MANIFEST.parent, cache_dir=paths.cache / 'cp1', progress=None)
    EXPECTED = dry['crops']['edited_crop_failed']['failed_1_90_rule']
else:
    from eviground.data.download import stage
    staged = stage('diffseg30k', paths=paths)
    print('DiffSeg30k on the local disk:', staged)
    assert MANIFEST.exists(), f'{MANIFEST} not found: it is written by notebook 01 (CP1 run 4)'
    CROPS, EXPECTED = 3000, RUN4_FAILED

In [ ]:
# Cell 4: rebuild the failed draws, check every image, save the record
import datetime
from eviground.data.crop_diagnostic import run_diagnostic

diag = run_diagnostic(MANIFEST, crops_per_class=CROPS, seed=42, expected_failures=EXPECTED,
                      workers=WORKERS)
record = {
    'diagnostic': 'edited crops (CP1 run 4)',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY, 'branch': BRANCH, 'session': info,
    'wall_time_h': round((time.time() - T0) / 3600, 3),
    'result': diag,
}
out = paths.runs / 'checkpoints' / 'crop_diagnostic.json'
out.write_text(json.dumps(record, indent=1, default=str))
print('saved', out, '| wall time', round((time.time() - T0) / 60, 1), 'min')
d, im = diag['draws'], diag['images']
print('reproduced run 4:', d['reproduced'], f"({d['failed']} of {d['edited_draws']} failed, expected {d['expected_failed']})")
print('failed draws by kind:', d['failed_by_kind'])
print('edited share of every failed try:', d['failed_tries_share_histogram'])
print('below 1%: edited pixels in the best try:', d['failed_below_1pct_best_try_edited_pixels'])
print('training rule failures on the same draws: 1% minimum', d['training_rule_failed_1pct'],
      '| 256-pixel minimum', d['training_rule_failed_min_256px'])
print('edited training images:', im['edited_training_images'], '| never a valid crop: 1-90%',
      im['never_valid_1_90pct'], '| 256 px-90%', im['never_valid_min_256px_le_90pct'])
print('whole-image edited share:', im['edited_share'])